# Nemotron 3 Nano - MinMax SFT (RTX 6000 Pro, Kaggle compute-safe)

Supervised fine-tuning with a selectable loss (mean NLL / MinMax worst-token / blend).

**Compute-safety design (96 GB GPU, ~12 h wall, locked weekly quota):**
- Memory-safe loss: fused `cross_entropy` (no fp32 full-vocab log_softmax spike).
- `SMOKE_TEST` first: 64 rows + 8 steps (~10 min) to prove the run before burning quota.
- Subset + 1 epoch + `TRAIN_MAX_LEN` cap keep the real run inside the session limit.
- `save_steps=100` + auto-resume so a crash loses <=100 steps.
- Vanilla TRL `SFTTrainer` (Unsloth's fused loss double-projects lm_head on
  Nemotron-H -> shape crash).
- One generalized system prompt for all 7 puzzle categories.


In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE, "USE_PRETRAINED": USE_PRETRAINED})

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)

import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

In [ ]:
import os

BASE_MODEL_NAME   = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
SFT_DATA_PATH     = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"

OUTPUT_ROOT       = "outputs"
SFT_ADAPTER_DIR   = os.path.join(OUTPUT_ROOT, "minmax_sft_adapter")
SUBMISSION_DIR    = os.path.join(OUTPUT_ROOT, "submission_minmax_sft")
TB_LOG_DIR        = os.path.join(OUTPUT_ROOT, "tb_logs_minmax_sft")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(TB_LOG_DIR, exist_ok=True)

SEED = 42

# ===========================================================================
# COMPUTE-SAFETY KNOBS  (Kaggle: 96 GB GPU, ~12 h wall, locked weekly quota)
# ===========================================================================
# 1) SMOKE_TEST=1 -> tiny dataset + few steps. ALWAYS run FIRST interactively
#    (~10 min). Flip to 0 only after it's green.
SMOKE_TEST  = 1
SMOKE_ROWS  = 64
SMOKE_STEPS = 8

# 2) Real-run scope. Subset + 1 epoch keeps wall-clock under the session limit.
SUBSET_N    = 3000      # None = all rows
NUM_EPOCHS  = 1

# 3) Sequence caps. Model loads at MODEL_MAX_LEN (capacity); TRAINING truncates
#    to TRAIN_MAX_LEN -> biggest lever on memory AND speed. Check the histogram
#    in the dataset cell and lower TRAIN_MAX_LEN if p99 is well below it.
MODEL_MAX_LEN = 8192
TRAIN_MAX_LEN = 4096

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print({"SMOKE_TEST": SMOKE_TEST, "SUBSET_N": SUBSET_N, "NUM_EPOCHS": NUM_EPOCHS,
       "MODEL_MAX_LEN": MODEL_MAX_LEN, "TRAIN_MAX_LEN": TRAIN_MAX_LEN})

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )

    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MODEL_MAX_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")

## LoRA Targets (RSLoRA r=32, NVIDIA-side priority)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
import re

linear_modules = []
for name, mod in model.named_modules():
    if mod.__class__.__name__ in ("Linear", "Linear4bit", "Linear8bitLt"):
        linear_modules.append(name)

target_regex = (
    r".*("
    r"self_attn\.(q|k|v|o)_proj"
    r"|mamba\.(in|out|x|dt|gate)_proj"
    r"|shared_experts\.(gate|up|down)_proj"
    r")$"
)
matched = [n for n in linear_modules if re.match(target_regex, n)]
print(f"LoRA target regex matched {len(matched)} modules.")
if len(matched) == 0:
    sample = [n for n in linear_modules if "expert" in n or "mamba" in n or "self_attn" in n][:20]
    raise RuntimeError(
        "LoRA target_regex matched 0 modules. NO silent fallback. "
        f"Sample module names for debugging: {sample}"
    )

lora_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
    target_modules=target_regex,
    task_type=TaskType.CAUSAL_LM,
    use_rslora=True,
    use_dora=True,    # Weight-Decomposed LoRA -- separates magnitude/direction; +2-4pp typical on reasoning
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

# --- audit: confirm targets attached to expected modules. Catches the
# "regex matched 0 -> suffix fallback attached LoRA to all 128 routable experts"
# failure mode locally instead of at eval time. See work/2026-05-29_worklog.md.
trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
n_train = sum(s for _, s in trainable)
print(f"[audit] {len(trainable)} trainable param tensors  total={n_train/1e6:.1f}M")
print(f"[audit] first 5: {[n for n,_ in trainable[:5]]}")
print(f"[audit] last  5: {[n for n,_ in trainable[-5:]]}")
assert 50_000_000 <= n_train <= 400_000_000, (
    f"[audit] trainable param count {n_train/1e6:.1f}M outside [50M, 400M] -- "
    "either regex missed everything (too low) or attached to routable experts "
    "(too high). Inspect target_regex against linear_modules above."
)

## Generalized System Prompt

One prompt for all categories (bit manipulation, number-base conversion,
gravitational constant, unit conversion, text encryption/cipher, algebra &
equation transformation). Fixes the `<think>...</think>` then `\boxed{}`
contract. Used identically at train + eval.


In [ ]:
SYSTEM_PROMPT = """You are a meticulous reasoning engine for a logical-puzzle benchmark. \
Every task has exactly one correct, deterministic answer that can be derived by \
exact rule-following and arithmetic. Speed does not matter; correctness does.

GENERAL METHOD
1. Read the problem twice. Identify the category and restate, in your own words, \
the exact transformation or quantity being asked for.
2. Extract every given value, rule, mapping, base, unit, and constant verbatim. \
Never invent data that is not stated.
3. Work strictly step by step. Show each intermediate result. Do arithmetic \
digit by digit and re-check it. When a rule is defined in the prompt, apply it \
literally rather than relying on prior assumptions.
4. Verify: substitute your answer back into the problem and confirm it satisfies \
every stated condition. If it does not, find your error and redo the step.

CATEGORY-SPECIFIC RULES
- Bit manipulation: operate on the stated bit width (default 8 bits). Preserve \
leading zeros. Apply AND/OR/XOR/NOT/shifts exactly; for shifts state whether bits \
fall off or wrap as specified. Report the result in the format the prompt uses \
(binary string, or decimal if asked).
- Number-base conversion: convert through base 10 as an intermediate when helpful. \
Map digits A-F (and beyond) carefully. State the source and target base. Do not \
add base prefixes unless asked.
- Gravitational constant / free-fall: use the constant exactly as given in the \
prompt (do not substitute a textbook g). Track units through every step. Round \
only at the end, to the precision the prompt implies.
- Unit conversion: write the conversion factor as an explicit fraction, cancel \
units, and keep full precision until the final rounding. State the final unit.
- Text encryption / cipher: determine the exact scheme (shift/Caesar, substitution, \
keyword, etc.) and direction (encrypt vs decrypt). Transform one character at a \
time, preserving case, spacing, and punctuation unless told otherwise.
- Algebraic equations & equation transformation: isolate the target symbol with \
inverse operations applied to both sides; or, if a transformation rule is defined, \
apply that exact rule. Keep equations balanced at every line.

OUTPUT CONTRACT (mandatory)
- First think inside a single <think> ... </think> block containing your full \
step-by-step derivation and verification.
- Immediately after </think>, output the final answer once, wrapped exactly as \
\\boxed{...} with nothing after it.
- The boxed content must be only the answer in the form the problem expects \
(e.g. an 8-bit binary string, a number, a word, or an expression) - no units \
unless the problem asks for them, no extra words."""

print(f"SYSTEM_PROMPT chars: {len(SYSTEM_PROMPT)}")
print(SYSTEM_PROMPT[:300], "...")

## SFT Dataset Prep (system + user + assistant, assistant-only masking)

Scope controlled by SMOKE_TEST / SUBSET_N. System + user tokens masked to -100;
loss sees only the assistant response.


In [ ]:
import pandas as pd, re
from datasets import Dataset as HFDataset

df_sft = pd.read_csv(SFT_DATA_PATH)
print(f"SFT data: {len(df_sft)} rows.  Columns: {list(df_sft.columns)}")

def _find(cols, names):
    low = {c.lower(): c for c in cols}
    for n in names:
        if n in low:
            return low[n]
    return None

PROMPT_COL = _find(df_sft.columns, ["prompt", "question", "problem", "input"])
ANSWER_COL = _find(df_sft.columns, ["answer", "solution", "label", "target", "final_answer"])
COT_COL    = _find(df_sft.columns, ["cot", "reasoning", "think", "generated_cot",
                                    "response", "completion", "rationale", "output"])
print(f"Detected -> prompt={PROMPT_COL!r}  answer={ANSWER_COL!r}  cot={COT_COL!r}")
if PROMPT_COL is None:
    raise ValueError(f"No prompt-like column in {list(df_sft.columns)}")

df_sft = df_sft.dropna(subset=[PROMPT_COL]).reset_index(drop=True)
df_sft = df_sft.sample(frac=1, random_state=SEED).reset_index(drop=True)

if SMOKE_TEST:
    df_sft = df_sft.head(SMOKE_ROWS).reset_index(drop=True)
    print(f"[SMOKE] using {len(df_sft)} rows")
elif SUBSET_N is not None:
    df_sft = df_sft.head(SUBSET_N).reset_index(drop=True)
    print(f"[REAL] using subset of {len(df_sft)} rows")
else:
    print(f"[REAL] using all {len(df_sft)} rows")


def _strip_boxed(text):
    r"""Remove every \boxed{...} via BRACE-BALANCED matching.

    The old `re.sub(r'\\boxed\{[^{}]*\}', ...)` could not match a box whose
    CONTENT contains a brace (these symbolic/cipher answers are literally things
    like `(/&{`). Such inline boxes survived, then the wrapper added a second
    \boxed{} -> count==2 -> the format check raised "malformed assistant target".
    Here we walk braces so any inline box is fully removed regardless of content.
    """
    tok = "\\boxed{"
    out, i = [], 0
    while i < len(text):
        j = text.find(tok, i)
        if j == -1:
            out.append(text[i:]); break
        out.append(text[i:j])
        k = j + len(tok); depth = 1
        while k < len(text) and depth > 0:
            if text[k] == "{": depth += 1
            elif text[k] == "}": depth -= 1
            k += 1
        i = k                      # skip the whole (possibly unbalanced) box
    return "".join(out)


def build_assistant_text(row):
    r"""Canonical target: <think>\n{reasoning}\n</think>\n\boxed{ans}.

    `ans` is taken verbatim from the answer column (may itself contain braces /
    symbols -- that's fine, it's the literal expected output).
    """
    ans = "" if ANSWER_COL is None else str(row[ANSWER_COL]).strip()
    cot = "" if COT_COL is None else str(row.get(COT_COL, "") or "").strip()

    # control the tags ourselves
    cot = cot.replace("<think>", "").replace("</think>", "").strip()
    # drop boilerplate "I will put/return my final answer inside \boxed{}" lines
    cot = re.sub(r'(?im)^.*I will (now )?(put|return) .*\\boxed\{\}.*$', '', cot)
    # drop trailing "The answer ... is \boxed{...}" lines
    cot = re.sub(r'(?im)^.*The answer .*\\boxed.*$', '', cot)
    # brace-balanced removal of any remaining inline \boxed{...}
    cot = _strip_boxed(cot)
    cot = re.sub(r'\n{3,}', '\n\n', cot).strip()

    think = cot if cot else "Work through the problem step by step."
    return f"<think>\n{think}\n</think>\n\\boxed{{{ans}}}"


records = []
for _, row in df_sft.iterrows():
    records.append({
        "system": SYSTEM_PROMPT,
        "user": str(row[PROMPT_COL]) + PROMPT_SUFFIX,
        "assistant": build_assistant_text(row),
    })

# Format check (presence-based, brace-tolerant): a closing </think> must exist and
# a \boxed{ must appear AFTER the last </think>. We do NOT count boxes or require a
# trailing '}' -- answers can legitimately contain braces.
def _well_formed(a):
    if "</think>" not in a:
        return False
    tail = a.rsplit("</think>", 1)[-1]
    return "\\boxed{" in tail

_bad = [i for i, r in enumerate(records) if not _well_formed(r["assistant"])]
if _bad:
    raise ValueError(f"{len(_bad)} malformed assistant targets (e.g. idx {_bad[:5]}). "
                     f"First: {records[_bad[0]]['assistant'][-200:]!r}")
print(f"[format] all {len(records)} targets well-formed: <think>..</think> then \\boxed{{}}")

raw_ds = HFDataset.from_list(records)
print(f"SFT records: {len(records)}")
print("\n--- sample assistant target HEAD ---\n", records[0]["assistant"][:300])
print("\n--- sample assistant target TAIL ---\n", records[0]["assistant"][-160:])

In [ ]:
def tokenize_with_assistant_mask(example):
    full_msgs = [
        {"role": "system",    "content": example["system"]},
        {"role": "user",      "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    prefix_msgs = [
        {"role": "system", "content": example["system"]},
        {"role": "user",   "content": example["user"]},
    ]

    def render(msgs, add_gen):
        try:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen, enable_thinking=True)
        except TypeError:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen)

    full_text   = render(full_msgs, False)
    prefix_text = render(prefix_msgs, True)

    full_ids   = tokenizer(full_text, add_special_tokens=False, truncation=True,
                           max_length=TRAIN_MAX_LEN)["input_ids"]
    prefix_ids = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]

    cutoff = min(len(prefix_ids), len(full_ids))
    labels = list(full_ids)
    for i in range(cutoff):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels}


tokenized_ds = raw_ds.map(
    tokenize_with_assistant_mask,
    remove_columns=raw_ds.column_names,
    desc="Tokenize + assistant mask",
)

def _has_label(ex):
    return any(t != -100 for t in ex["labels"])
before = len(tokenized_ds)
tokenized_ds = tokenized_ds.filter(_has_label)
print(f"Kept {len(tokenized_ds)}/{before} rows with >=1 unmasked assistant token.")

import numpy as np
_lens = np.array([len(x) for x in tokenized_ds["input_ids"]])
pct = lambda p: int(np.percentile(_lens, p))
print(f"Token length  min={_lens.min()}  mean={_lens.mean():.0f}  "
      f"p50={pct(50)}  p90={pct(90)}  p99={pct(99)}  max={_lens.max()}")
print(f"TRAIN_MAX_LEN={TRAIN_MAX_LEN}. If p99 << this, lower it + re-run this cell "
      f"to save memory + time.")

In [ ]:
import torch

class CompletionOnlyDataCollator:
    """Pads input_ids/labels/attention_mask; preserves -100 on prompt tokens."""
    def __init__(self, tokenizer, label_pad_id=-100):
        self.tok = tokenizer
        self.pad_id = tokenizer.pad_token_id
        self.label_pad_id = label_pad_id

    def __call__(self, features):
        maxlen = max(len(f["input_ids"]) for f in features)
        input_ids, labels, attn = [], [], []
        for f in features:
            ids = list(f["input_ids"])
            lab = list(f["labels"])
            pad = maxlen - len(ids)
            input_ids.append(ids + [self.pad_id] * pad)
            labels.append(lab + [self.label_pad_id] * pad)
            attn.append([1] * len(ids) + [0] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

data_collator = CompletionOnlyDataCollator(tokenizer)
print("Collator ready.")

## Vanilla TRL + Memory-Safe Loss

Unsloth's patched `SFTTrainer` runs a fused loss that re-projects `lm_head` and
assumes a pure-Transformer hidden dim -> crashes on Nemotron-H. We purge Unsloth
and subclass the vanilla `SFTTrainer`. The loss uses fused `cross_entropy`
(no fp32 full-vocab `log_softmax` -> that ~4.3 GB/step spike was the OOM cause).


In [ ]:
import os, sys
os.environ["TORCHDYNAMO_DISABLE"]   = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

import torch
import torch._dynamo
torch._dynamo.config.disable = True
torch._dynamo.reset()

for _m in list(sys.modules):
    if _m == "trl" or _m.startswith("trl.") or "unsloth" in _m.lower():
        del sys.modules[_m]
sys.meta_path = [f for f in sys.meta_path
                 if "unsloth" not in type(f).__module__.lower()]

import torch.nn.functional as F
from trl import SFTTrainer, SFTConfig
assert "unsloth" not in SFTTrainer.__module__.lower(), \
    f"Still using Unsloth trainer: {SFTTrainer.__module__}"
print(f"SFTTrainer module: {SFTTrainer.__module__}  (vanilla TRL, dynamo disabled)")

# "mean" = standard NLL (SAFE, proven, default). "minmax" = worst-token objective.
# "blend" = 0.5*mean + 0.5*minmax. Recommend mean for the first real run.
LOSS_MODE = "mean"   # "mean" | "minmax" | "blend"
print("LOSS_MODE =", LOSS_MODE)


class MinMaxSFTTrainer(SFTTrainer):
    """Selectable, memory-safe SFT loss.

    Fused `F.cross_entropy(reduction="none")` gives -log p(true) WITHOUT a full
    fp32 (seq x vocab) log_softmax tensor. `**kwargs` swallows num_items_in_batch.
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits                       # (B, T, V) bf16

        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:].to(shift_logits.device)
        B, T, V = shift_logits.shape

        nll_flat = F.cross_entropy(
            shift_logits.reshape(-1, V),
            shift_labels.reshape(-1),
            ignore_index=-100,
            reduction="none",
        ).view(B, T)                                  # -log p(true), 0 where ignored
        mask = (shift_labels != -100)

        if LOSS_MODE == "mean":
            loss = nll_flat[mask].mean()
        else:
            token_logp = (-nll_flat).masked_fill(~mask, float("inf"))
            min_logp, _ = token_logp.min(dim=-1)      # worst assistant token / seq
            valid = mask.any(dim=-1)
            minmax = -min_logp[valid].mean()
            if LOSS_MODE == "minmax":
                loss = minmax
            elif LOSS_MODE == "blend":
                loss = 0.5 * nll_flat[mask].mean() + 0.5 * minmax
            else:
                raise ValueError(f"bad LOSS_MODE={LOSS_MODE}")

        if not torch.isfinite(loss):
            loss = (logits.sum() * 0.0).requires_grad_(True)  # skip nan/inf step
        return (loss, outputs) if return_outputs else loss


print("MinMaxSFTTrainer defined.")

## SFT Config (NVIDIA-side optimizer + settings)

In [ ]:
_max_steps = SMOKE_STEPS if SMOKE_TEST else -1   # -1 = use epochs

sft_config = SFTConfig(
    output_dir                   = os.path.join(OUTPUT_ROOT, "minmax_sft_run"),
    num_train_epochs             = NUM_EPOCHS,
    max_steps                    = _max_steps,
    per_device_train_batch_size  = 1,
    gradient_accumulation_steps  = 8,
    learning_rate                = 8e-5,
    lr_scheduler_type            = "cosine",
    warmup_ratio                 = 0.05,
    weight_decay                 = 0.01,
    max_grad_norm                = 1.0,
    optim                        = "paged_adamw_8bit",
    adam_beta1                   = 0.9,
    adam_beta2                   = 0.999,
    bf16                         = True,
    gradient_checkpointing       = True,
    gradient_checkpointing_kwargs= {"use_reentrant": True},
    max_length                   = TRAIN_MAX_LEN,
    packing                      = False,
    dataset_kwargs               = {"skip_prepare_dataset": True},
    remove_unused_columns        = False,
    logging_steps                = 1 if SMOKE_TEST else 5,
    logging_dir                  = TB_LOG_DIR,
    report_to                    = "none",
    save_strategy                = "no" if SMOKE_TEST else "steps",
    save_steps                   = 100,
    save_total_limit             = 2,
    seed                         = SEED,
    dataloader_num_workers       = 2,
)
print(f"SFTConfig ready. mode={'SMOKE' if SMOKE_TEST else 'REAL'}  "
      f"max_steps={_max_steps}  epochs={NUM_EPOCHS}  "
      f"eff_batch={sft_config.per_device_train_batch_size*sft_config.gradient_accumulation_steps}  "
      f"max_length={TRAIN_MAX_LEN}")

## Launch (smoke first, then real)

In [ ]:
import gc, time, torch, os, glob, shutil

trainer = MinMaxSFTTrainer(
    model           = model,
    args            = sft_config,
    train_dataset   = tokenized_ds,
    data_collator   = data_collator,
    processing_class= tokenizer,
)

_ckpts = sorted(glob.glob(os.path.join(sft_config.output_dir, "checkpoint-*")),
                key=lambda p: int(p.rsplit("-", 1)[-1]))
resume = bool(_ckpts) and not SMOKE_TEST
print(f"{'Resuming from' if resume else 'Fresh start; no'} checkpoint in {sft_config.output_dir}")

torch.cuda.empty_cache(); gc.collect()
torch.cuda.reset_peak_memory_stats()
t0 = time.time()

train_err = None
try:
    trainer.train(resume_from_checkpoint=resume)
    print(f"Training done in {(time.time()-t0)/60:.1f} min")
except Exception as e:
    train_err = e
    print(f"[TRAIN ERROR after {(time.time()-t0)/60:.1f} min] {type(e).__name__}: {e}")
    print("[recovery] will still try to save whatever progressed so far.")

print(f"PEAK VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")


def _save_adapter(dest):
    """Save LoRA adapter robustly. Tries PEFT-aware save first, then trainer
    save, then copies from latest checkpoint dir. Verifies required files."""
    os.makedirs(dest, exist_ok=True)
    needed = ["adapter_config.json", "adapter_model.safetensors"]

    # 1) PEFT-aware: the trainer's underlying model is the PeftModel (canonical
    # adapter-save path, handles Unsloth-wrapped models too).
    try:
        inner = trainer.model
        if hasattr(inner, "save_pretrained"):
            inner.save_pretrained(dest)
        tokenizer.save_pretrained(dest)
    except Exception as e:
        print(f"[save] inner.save_pretrained failed: {e}; trying trainer.save_model")
        try:
            trainer.save_model(dest)
            tokenizer.save_pretrained(dest)
        except Exception as e2:
            print(f"[save] trainer.save_model also failed: {e2}")

    # 2) verify -> if still missing, copy from latest checkpoint dir
    missing = [n for n in needed if not os.path.exists(os.path.join(dest, n))]
    if missing:
        ckpts = sorted(glob.glob(os.path.join(sft_config.output_dir, "checkpoint-*")),
                       key=lambda p: int(p.rsplit("-", 1)[-1]))
        if ckpts:
            src = ckpts[-1]
            print(f"[save] {missing} missing in {dest}; copying from {src}")
            for fname in needed:
                sp = os.path.join(src, fname)
                if os.path.exists(sp):
                    shutil.copy2(sp, os.path.join(dest, fname))

    # 3) final verify
    have = {n: os.path.exists(os.path.join(dest, n)) for n in needed}
    sizes = {n: (os.path.getsize(os.path.join(dest, n)) / 1024 / 1024 if have[n] else 0)
             for n in needed}
    print(f"[save] {dest} -> have={have}  sizes_MB={ {k: f'{v:.1f}' for k, v in sizes.items()} }")
    return all(have.values())


# Always save -- smoke runs need an inspectable artifact too, real runs need it
# even if train() exited noisily.
ok = _save_adapter(SFT_ADAPTER_DIR)
if not ok:
    print("[save] WARNING: required files still missing. Available checkpoint dirs:",
          glob.glob(os.path.join(sft_config.output_dir, "checkpoint-*")))
else:
    print(f"Adapter saved + verified -> {SFT_ADAPTER_DIR}")

if SMOKE_TEST:
    print("\n[SMOKE] green if no OOM/nan + PEAK VRAM has headroom.")

if train_err is not None:
    raise train_err

## Greedy Sanity Check (confirm \\boxed{} emitted)

In [ ]:
import torch

model.eval()
_probe = raw_ds[0]["user"]
_msgs = [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": _probe}]
try:
    _txt = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
except TypeError:
    _txt = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)

_inputs = tokenizer(_txt, return_tensors="pt").to(model.device)
with torch.no_grad():
    _out = model.generate(**_inputs, max_new_tokens=512, do_sample=False, temperature=None, top_p=None)
_gen = tokenizer.decode(_out[0][_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(_gen[:1200])
print("\nHAS_BOXED:", "\\boxed{" in _gen)
model.train()

## Package submission.zip

In [ ]:
import json, shutil, zipfile, os, glob, sys, subprocess

needed = ["adapter_config.json", "adapter_model.safetensors"]
src_dir = SFT_ADAPTER_DIR

# Kaggle eval reads submission.zip from /kaggle/working root, NOT from a subdir.
# Old packaging wrote inside OUTPUT_ROOT -> Kaggle never found it. Write to
# working-root explicitly; fall back to OUTPUT_ROOT off-Kaggle.
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else OUTPUT_ROOT
print(f"[package] WORKING = {WORKING}")

def _have_all(d):
    return all(os.path.exists(os.path.join(d, n)) for n in needed)

# 1) Recovery from checkpoint dir if main missing
ckpts = []
if not _have_all(src_dir):
    print(f"[package] {needed} not all present in {src_dir}")
    output_dir = sft_config.output_dir if "sft_config" in dir() else None
    ckpts = sorted(
        glob.glob(os.path.join(output_dir, "checkpoint-*")) if output_dir else [],
        key=lambda p: int(p.rsplit("-", 1)[-1]),
    )
    if ckpts:
        ck = ckpts[-1]
        print(f"[package] copying from latest checkpoint: {ck}")
        os.makedirs(src_dir, exist_ok=True)
        for fname in needed:
            sp = os.path.join(ck, fname)
            if os.path.exists(sp):
                shutil.copy2(sp, os.path.join(src_dir, fname))

    if not _have_all(src_dir) and "trainer" in dir():
        print(f"[package] still missing -- attempting trainer.model.save_pretrained({src_dir})")
        try:
            trainer.model.save_pretrained(src_dir)
            tokenizer.save_pretrained(src_dir)
        except Exception as e:
            print(f"[package] live save failed: {e}")

missing = [n for n in needed if not os.path.exists(os.path.join(src_dir, n))]
if missing:
    raise FileNotFoundError(
        f"Adapter files {missing} still missing after recovery attempts.\n"
        f"  Checked: {src_dir}\n"
        f"  Available checkpoints: {ckpts if ckpts else 'none'}\n"
        f"  Re-run the launch cell to retrain + save, or manually copy a checkpoint."
    )

# 2) Post-processing surgery: rename keys + unfuse experts + fuse Mamba (optional
# SV-amplify). Reads src_dir, writes vLLM-clean adapter to processed_dir. Output
# of this step is what we zip. Override boost via env: POSTPROC_BOOST=1.12 for
# the SV-amplification A/B test (top 50% singular values * 1.12). Default 1.0.
POSTPROC_BOOST = float(os.environ.get("POSTPROC_BOOST", "1.0"))
POSTPROC_TOP_FRAC = float(os.environ.get("POSTPROC_TOP_FRAC", "0.5"))
processed_dir = src_dir + "_processed"

script_path = None
for cand in ["tools/postprocess_adapter.py",
             "/kaggle/working/tools/postprocess_adapter.py",
             os.path.join(os.getcwd(), "tools", "postprocess_adapter.py")]:
    if os.path.exists(cand):
        script_path = cand
        break

if script_path:
    cmd = [sys.executable, script_path,
           "--in",  src_dir,
           "--out", processed_dir,
           "--boost", str(POSTPROC_BOOST),
           "--top-frac", str(POSTPROC_TOP_FRAC),
           "--rank", "32"]
    print(f"[package] running adapter post-processing: {' '.join(cmd)}")
    rc = subprocess.run(cmd, check=False).returncode
    if rc != 0 or not all(os.path.exists(os.path.join(processed_dir, n)) for n in needed):
        print(f"[package] post-processing failed (rc={rc}); falling back to raw adapter")
        processed_dir = src_dir
else:
    print(f"[package] tools/postprocess_adapter.py NOT FOUND; using raw adapter "
          f"(submission may fail at eval if module names do not match vLLM)")
    processed_dir = src_dir

# 3) Copy chosen adapter (processed or raw fallback) -> submission dir + final patch
os.makedirs(SUBMISSION_DIR, exist_ok=True)
for fname in needed:
    sp = os.path.join(processed_dir, fname)
    dp = os.path.join(SUBMISSION_DIR, fname)
    shutil.copy2(sp, dp)
    print(f"  copied {fname}  ({os.path.getsize(dp)/1024/1024:.1f} MB)")

cfg_path = os.path.join(SUBMISSION_DIR, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

# 4) Write zip at WORKING root (Kaggle convention) so eval picks it up.
zip_path = os.path.join(WORKING, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in needed:
        zf.write(os.path.join(SUBMISSION_DIR, fname), fname)
print(f"\nsubmission.zip: {zip_path}  ({os.path.getsize(zip_path)/1024/1024:.1f} MB) - ready "
      f"(boost={POSTPROC_BOOST}  top_frac={POSTPROC_TOP_FRAC}).")